# 01 — DAM baselines: seasonal naive on ESIOS-ES (2024)

First milestone of sub-objective 3.1. Establishes the **seasonal naive
benchmark** that every more elaborate model (LEAR, DNN) must beat
before being declared useful (Lago et al. 2021, §3.1).

**Setup**
- Target: `price_es` pulled directly from ESIOS indicator 600 geo=3
  (`Precio mercado SPOT Diario`, España). This is the canonical
  wholesale Spanish DAM price. The data is cached locally by month
  under `data/cache/esios/` so subsequent runs are instantaneous.
- Training window: 365 days, recalibrated daily (rolling).
- Test horizon: 2024-01-01 → 2024-12-31.
- Metric: MAE and sMAPE (Lago 2021 form); rMAE is reported in
  subsequent notebooks once we have a non-naive model to compare.

**Why this baseline matters.** The weekly seasonal naive captures the
daily profile and the day-of-week effect (weekday/weekend) without
any parameters. Any model that does not produce a meaningfully lower
MAE is not learning anything useful about the price formation process.

**Note on data sourcing.** Earlier iterations used `price_es` from the
v8 parquet of `mibel-congestion-monitor`. A diagnostic in the same
session (`reports/diagnostics/v8_vs_esios_2024.parquet`) showed the v8
panel has its ES / PT labels crossed at source: v8.price_es matches
ESIOS-PT (Portugal / MIBEL-coupled) at 99.66% in 2024, while
v8.price_fr matches ESIOS-ES at 99.66%. The DAM loader now bypasses
the v8 price columns entirely and pulls ESIOS 600 directly.

In [ ]:
from __future__ import annotations
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from mibel_forecasting.data.loaders import load_dam_panel
from mibel_forecasting.evaluation.metrics import by_hour, mae, smape
from mibel_forecasting.evaluation.recalibration import rolling_forecast
from mibel_forecasting.models.naive import SeasonalNaive

REPORTS = Path('../reports/figures').resolve()
REPORTS.mkdir(parents=True, exist_ok=True)

## 1. Load the DAM panel

We pull ESIOS 600 (España, Portugal, Francia) for 2022 → 2024 so the
rolling 365-day window has a full year of warm-up before the 2024 test
horizon. `v8_exogenous=None` keeps the loader ESIOS-only for this
naive baseline; later notebooks will join in the v8 demand / wind /
solar / gas / CO₂ features.

In [ ]:
df = load_dam_panel(start='2022-01-01', end='2024-12-31', v8_exogenous=None)
print(f'shape={df.shape}, tz={df.index.tz}')
print(f'range: {df.index.min()} -> {df.index.max()}')
df.head(3)

## 2. Run the rolling seasonal naive (recalibrated daily)

365-day window, step 1 day, test horizon = full year 2024.

In [ ]:
forecasts = rolling_forecast(
    df,
    target_col='price_es',
    model_factory=lambda: SeasonalNaive(target_col='price_es'),
    train_size='365D',
    test_start='2024-01-01',
    test_end='2024-12-31',
)
forecasts.tail(3)

## 3. Aggregate metrics

MAE in €/MWh and sMAPE (Lago 2021 form, [0, 200]%).

In [ ]:
agg = pd.Series({
    'MAE (EUR/MWh)': mae(forecasts['y_true'], forecasts['y_pred']),
    'sMAPE (%)':     smape(forecasts['y_true'], forecasts['y_pred']),
    'N (hours)':     forecasts.dropna().shape[0],
}, name='SeasonalNaive (lag=7d)')
agg

## 4. Figure — MAE by hour of day

One figure, one question: *where does the naive baseline lose the
most money?* The hours where MAE is largest are where a learned model
has the most room to add value; they are also a sanity check (peaks
should fall on the evening ramp-down, when prices swing fastest).

In [ ]:
hourly_mae = by_hour(forecasts['y_true'], forecasts['y_pred'], mae)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(hourly_mae.index, hourly_mae.values, color='#4C72B0')
ax.set_xlabel('Hour of day (Europe/Madrid)')
ax.set_ylabel('MAE (EUR/MWh)')
ax.set_title('Seasonal naive (lag=7d) - MAE by hour, DAM ES 2024 (ESIOS 600)')
ax.set_xticks(range(0, 24, 2))
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(REPORTS / '01_naive_mae_by_hour.png', dpi=150)
plt.show()
hourly_mae.round(2).to_frame('MAE')

## 5. Discussion

The seasonal naive on the 2024 Spanish DAM (ESIOS-ES) yields MAE
≈ 27 €/MWh and sMAPE ≈ 77% — this is the order of magnitude any LEAR
or DNN model has to beat to justify its complexity. Two reads worth
carrying into the next notebook:

1. The error profile is **not flat across hours**. The peak at the
   evening ramp-down (~20h) reflects the transition between
   solar-dominated mid-day prices and thermal-dominated off-peaks.
   A learned model has the most room to improve precisely there.
2. The sMAPE is inflated by mid-day hours where prices approach zero
   under high solar penetration (Aineto et al. 2019). MAE is the more
   interpretable headline number for DAM-ES under the current
   generation mix; sMAPE is kept only for compatibility with Lago
   2021. The next notebook will introduce **rMAE** (relative to this
   naive), which cancels the small-denominator inflation by comparing
   model-vs-naive on the same realised series.

**Effect of the v8 label fix.** The same naive run on the
(incorrectly labelled) v8.price_es series — which is actually
MIBEL-coupled, not pure Spain — produced MAE ≈ 26.9. With the correct
ES-only target the MAE is slightly *higher* because Spain alone is
more volatile (~6% of hours at exactly 0 €/MWh; the Iberian-coupled
series is smoothed by Portugal). That leaves *more* room for a
learned model to add value, which is metodologically the honest
starting point.

**Next** — notebook 02 adds technical indicators (Demir 2019) on top
of the same panel and reports the relative MAE versus this naive.